# Hotel Booking Cancellation Prediction – K-Nearest Neighbours

**Prerequisite:** run `FDM_Preprocessing_KNN_RF.ipynb` (Parts A & B) first. It writes the preprocessed matrices to `data/` and the fitted encoders to `models/preprocessors.joblib`.

KNN is distance-based, so it uses the **scaled** feature matrix (`X_*_scaled.csv`): winsorised and standardised numerics, the ordinal lead-time band and one-hot categories, all fitted on the training set only.

**Validation protocol (from the proposal):**
- Cross-validation uses `StratifiedGroupKFold` with the same predictor-hash groups as the train/test split, so identical bookings never sit on both sides of a fold.
- Hyper-parameters are tuned on a stratified, group-aware **50% sample** of the training set, then the best setting is refit on the full training set.
- KNN has no `class_weight`, so imbalance is handled with **SMOTE-NC applied inside each training fold only**, and compared with no resampling.
- Tuning is scored by **PR-AUC**; recall, precision, F1, ROC-AUC and confusion matrices are also reported.
- The decision threshold is chosen from **out-of-fold training predictions** using a stated cost assumption. The test set is only used for the final evaluation.

**Optimisations for KNN (each one is measured by cross-validation below):**
1. **Redundant features removed.** Several columns carry the same information (e.g. `lead_time`, `lead_time_log`, `lead_time_band`). KNN adds up differences over all columns, so duplicated information is counted several times in every distance.
2. **Feature weighting.** Each feature is multiplied by a weight based on its mutual information with the target, so informative features count more in the distance. Weights are computed on the training fold only.
3. **Tuning on a larger sample** (50% instead of 25%) with a wider `k` range.

In [ ]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import StratifiedGroupKFold, ParameterGrid
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve, precision_recall_curve,
                             classification_report)

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
N_JOBS = -1

### LOAD PREPROCESSED DATA

In [ ]:
X_train_scaled = pd.read_csv("data/X_train_scaled.csv")
X_test_scaled = pd.read_csv("data/X_test_scaled.csv")
y_train = pd.read_csv("data/y_train.csv").squeeze("columns").astype(int)
y_test = pd.read_csv("data/y_test.csv").squeeze("columns").astype(int)
X_train_scaled_smote = pd.read_csv("data/X_train_scaled_smote.csv")
y_train_smote = pd.read_csv("data/y_train_smote.csv").squeeze("columns").astype(int)
meta_train = pd.read_csv("data/meta_train.csv", parse_dates=["arrival_date"])
meta_test = pd.read_csv("data/meta_test.csv", parse_dates=["arrival_date"])

train_groups_arr = meta_train["group"].to_numpy()
print("X_train_scaled:", X_train_scaled.shape, "| X_test_scaled:", X_test_scaled.shape)
print(f"Cancellation rate - train: {y_train.mean():.4f} | test: {y_test.mean():.4f}")

### FEATURE SET FOR KNN – remove redundant columns

Part B keeps several versions of the same information so that each model family can use the form it prefers. A tree can ignore duplicates, but KNN cannot: every duplicate adds to the distance again. One version of each is kept:

| Information | Kept | Dropped |
|---|---|---|
| Lead time | `lead_time_log` | `lead_time`, `lead_time_band` |
| Price | `adr_log` | `adr`, `adr_per_person` |
| Stay length | `stays_in_weekend_nights`, `stays_in_week_nights` | `total_nights` |
| Party size | `adults`, `children`, `babies` | `total_guests`, `is_family` |
| Arrival month | `arrival_month_sin`, `arrival_month_cos` | `arrival_date_week_number`, `arrival_season_*` |
| History / requests | the counts and `prev_cancellation_rate` | `has_previous_cancellation`, `has_parking_space`, `has_special_requests` |

In [ ]:
redundant_cols = (
    ["lead_time", "lead_time_band", "adr", "adr_per_person", "total_nights", "total_guests", "is_family",
     "arrival_date_week_number", "has_previous_cancellation", "has_parking_space", "has_special_requests"]
    + [c for c in X_train_scaled.columns if c.startswith("arrival_season_")]
)
all_features = list(X_train_scaled.columns)
knn_features = [c for c in all_features if c not in redundant_cols]
print(f"All features: {len(all_features)} | KNN feature set: {len(knn_features)} "
      f"({len(all_features) - len(knn_features)} redundant columns removed)")

# Binary (one-hot / flag) columns and the ordinal band: treated as categorical by SMOTE-NC
# and as discrete by the mutual-information estimator
discrete_cols = set(X_train_scaled.columns[X_train_scaled.nunique() <= 2]) | {"lead_time_band"}

### HELPERS – metrics, CV splitter, SMOTE-NC, feature weighting

**Feature weighting:** mutual information (MI) measures how much knowing a feature reduces uncertainty about cancellation. Each feature is multiplied by `(MI / max MI) ** power`. With `power = 0` every feature has weight 1 (plain KNN); a larger power makes the distance focus more on the informative features. `power` is tuned like any other hyper-parameter, and MI is always computed on the training fold only.

In [ ]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def evaluate(y_true, proba, threshold=0.5):
    """Standard metric set for the cancellation (positive) class."""
    pred = (proba >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred),
        "F1": f1_score(y_true, pred),
        "ROC-AUC": roc_auc_score(y_true, proba),
        "PR-AUC": average_precision_score(y_true, proba),
    }

def smote_nc(X, y, categorical_mask, k=5, random_state=42):
    """SMOTE-NC (same implementation as Part B): numeric columns are interpolated,
    binary/ordinal columns are copied from the base minority row."""
    rng = np.random.default_rng(random_state)
    counts = y.value_counts()
    minority_label = counts.idxmin()
    n_new = int(counts.max() - counts.min())

    cat_mask = np.asarray(categorical_mask, dtype=bool)
    num_idx, cat_idx = np.where(~cat_mask)[0], np.where(cat_mask)[0]
    X_min = X.loc[y == minority_label].to_numpy(dtype=float)

    penalty = np.median(X_min[:, num_idx].std(axis=0)) / np.sqrt(2)
    X_dist = np.hstack([X_min[:, num_idx], X_min[:, cat_idx] * penalty])
    neigh_full = NearestNeighbors(n_neighbors=k + 1).fit(X_dist).kneighbors(X_dist, return_distance=False)
    neigh = np.array([row[row != i][:k] for i, row in enumerate(neigh_full)])

    base = rng.integers(0, len(X_min), n_new)
    nbr = neigh[base, rng.integers(0, k, n_new)]
    gap = rng.random((n_new, 1))

    synth = X_min[base].copy()
    synth[:, num_idx] = X_min[base][:, num_idx] + gap * (X_min[nbr][:, num_idx] - X_min[base][:, num_idx])

    X_new = pd.DataFrame(synth, columns=X.columns)
    y_new = pd.Series(minority_label, index=X_new.index, name=y.name)
    X_out = pd.concat([X.reset_index(drop=True), X_new], ignore_index=True)
    y_out = pd.concat([y.reset_index(drop=True), y_new], ignore_index=True)
    order = rng.permutation(len(X_out))
    return X_out.iloc[order].reset_index(drop=True), y_out.iloc[order].reset_index(drop=True)

def mi_scores(X, y):
    """Mutual information between each feature and the target."""
    return pd.Series(
        mutual_info_classif(X, y, discrete_features=X.columns.isin(discrete_cols), random_state=RANDOM_STATE),
        index=X.columns)

def mi_to_weights(mi, power):
    """Feature weights in [0, 1]: (MI / max MI) ** power."""
    return ((mi / mi.max()).clip(lower=0) ** power).to_numpy()

def oof_knn(params, X, y, groups, use_smote=False, power=0.0, mi_cache=None):
    """Out-of-fold probabilities for KNN.
    SMOTE-NC (if used) and the MI feature weights (if power > 0) are fitted on the training fold only.
    mi_cache: optional dict reused across calls on the same data, so MI is computed once per fold."""
    oof = np.zeros(len(y))
    for fold, (tr, va) in enumerate(cv.split(X, y, groups=groups)):
        X_tr, y_tr, X_va = X.iloc[tr], y.iloc[tr], X.iloc[va]
        weights = None
        if power > 0:
            if mi_cache is not None and fold in mi_cache:
                mi = mi_cache[fold]
            else:
                mi = mi_scores(X_tr, y_tr)
                if mi_cache is not None:
                    mi_cache[fold] = mi
            weights = mi_to_weights(mi, power)
        if use_smote:
            X_tr, y_tr = smote_nc(X_tr, y_tr, X.columns.isin(discrete_cols), k=5, random_state=RANDOM_STATE)
        if weights is not None:
            X_tr, X_va = X_tr * weights, X_va * weights
        model = KNeighborsClassifier(**params, n_jobs=N_JOBS).fit(X_tr, y_tr)
        oof[va] = model.predict_proba(X_va)[:, 1]
    return oof

# Stratified, group-aware 50% tuning sample (one fold of a 2-fold group split)
tune_splitter = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=RANDOM_STATE)
_, tune_pos = next(tune_splitter.split(X_train_scaled, y_train, groups=train_groups_arr))
X_tune_all = X_train_scaled.iloc[tune_pos]          # all 93 features (for the "before" comparisons)
X_tune = X_tune_all[knn_features]                   # reduced KNN feature set
y_tune = y_train.iloc[tune_pos]
groups_tune = train_groups_arr[tune_pos]
print(f"Tuning sample: {len(tune_pos):,} rows | cancellation rate {y_tune.mean():.4f}")

### BASELINE KNN (sklearn defaults: k=5, uniform, Euclidean, all features, no weighting, no resampling)

In [ ]:
t0 = time.time()
knn_base_oof = oof_knn({"n_neighbors": 5}, X_tune_all, y_tune, groups_tune)
knn_baseline_cv = evaluate(y_tune, knn_base_oof)
print(f"KNN baseline CV ({time.time() - t0:.1f}s):")
print(pd.Series(knn_baseline_cv).round(4).to_string())

### HYPER-PARAMETER TUNING (grid search with group-aware 5-fold CV)

The grid covers the number of neighbours, the vote weighting, the distance metric and the feature-weighting power (`0` = no feature weighting).

In [ ]:
knn_grid = ParameterGrid({
    "n_neighbors": [15, 25, 35, 50, 75],
    "weights": ["uniform", "distance"],
    "p": [1, 2],               # 1 = Manhattan, 2 = Euclidean
    "power": [0, 0.35, 0.5],   # feature-weighting strength
})

mi_cache_tune = {}             # MI per fold, computed once and reused by every grid setting
t0 = time.time()
knn_rows = []
for g in knn_grid:
    params = {k: v for k, v in g.items() if k != "power"}
    oof = oof_knn(params, X_tune, y_tune, groups_tune, power=g["power"], mi_cache=mi_cache_tune)
    knn_rows.append({**g, **evaluate(y_tune, oof)})
knn_cv_results = pd.DataFrame(knn_rows).sort_values("PR-AUC", ascending=False).reset_index(drop=True)
print(f"KNN grid search: {len(knn_grid)} settings x 5 folds in {time.time() - t0:.1f}s")
knn_cv_results.round(4).head(10)

In [ ]:
best_knn_row = knn_cv_results.iloc[0]
knn_best_params = {"n_neighbors": int(best_knn_row["n_neighbors"]),
                   "weights": best_knn_row["weights"],
                   "p": int(best_knn_row["p"])}
knn_power = float(best_knn_row["power"])
print("Best KNN params:", knn_best_params, "| feature-weighting power:", knn_power)

# Effect of k for each feature-weighting power (best vote weighting / metric held fixed)
k_curve = knn_cv_results[(knn_cv_results["weights"] == knn_best_params["weights"]) &
                         (knn_cv_results["p"] == knn_best_params["p"])]
fig, ax = plt.subplots(figsize=(7, 4))
for power, grp in k_curve.groupby("power"):
    grp = grp.sort_values("n_neighbors")
    ax.plot(grp["n_neighbors"], grp["PR-AUC"], marker="o", label=f"power = {power:g}")
ax.set_xlabel("n_neighbors (k)")
ax.set_ylabel("CV PR-AUC")
ax.set_title("KNN: Cross-Validated PR-AUC vs k and Feature-Weighting Power", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

### CLASS IMBALANCE – SMOTE-NC inside folds vs no resampling (best setting)

In [ ]:
t0 = time.time()
smote_oof = oof_knn(knn_best_params, X_tune, y_tune, groups_tune, use_smote=True,
                    power=knn_power, mi_cache=mi_cache_tune)
smote_cmp = pd.DataFrame({
    "No resampling": best_knn_row[["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]].astype(float),
    "SMOTE-NC inside folds": pd.Series(evaluate(y_tune, smote_oof))[["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]],
})
knn_use_smote = bool(smote_cmp.loc["PR-AUC", "SMOTE-NC inside folds"] > smote_cmp.loc["PR-AUC", "No resampling"])
print(f"SMOTE-NC comparison in {time.time() - t0:.1f}s | use SMOTE-NC for the final model: {knn_use_smote}")
smote_cmp.round(4)

### EFFECT OF EACH OPTIMISATION STEP (cross-validated on the tuning sample)

Each row adds one change to the row above it, so the gain from every step is visible. "Previous tuned model" is the setting chosen before the optimisations (all features, k = 75, distance-weighted votes, Manhattan).

In [ ]:
prev_params = {"n_neighbors": 75, "weights": "distance", "p": 1}
steps = [
    ("0. Baseline (defaults, all features)", knn_baseline_cv),
    ("1. Previous tuned model (all features)", evaluate(y_tune, oof_knn(prev_params, X_tune_all, y_tune, groups_tune))),
    ("2. + redundant features removed", evaluate(y_tune, oof_knn(prev_params, X_tune, y_tune, groups_tune))),
    ("3. + feature weighting", evaluate(y_tune, oof_knn(prev_params, X_tune, y_tune, groups_tune,
                                                         power=knn_power or 0.5, mi_cache=mi_cache_tune))),
    ("4. + retuned k / weights / metric (final)", dict(best_knn_row[["Accuracy", "Precision", "Recall", "F1",
                                                                      "ROC-AUC", "PR-AUC"]].astype(float))),
]
optimisation_steps = pd.DataFrame({name: scores for name, scores in steps}).T
optimisation_steps.round(4)

### FINAL MODEL – refit on the full training set

In [ ]:
# Feature weights from the full training set (real rows only)
mi_full = mi_scores(X_train_scaled[knn_features], y_train)
feature_weights = mi_to_weights(mi_full, knn_power) if knn_power > 0 else np.ones(len(knn_features))

fig, ax = plt.subplots(figsize=(8, 5))
pd.Series(feature_weights, index=knn_features).sort_values().tail(15).plot(kind="barh", ax=ax, color="#ff7f0e")
ax.set_title("KNN Feature Weights (Top 15)", fontweight="bold")
ax.set_xlabel("Weight applied to the scaled feature")
plt.tight_layout()
plt.show()

# X_train_scaled_smote (Part B) is SMOTE-NC applied to the full training set, which is safe for the final fit.
if knn_use_smote:
    X_fit, y_fit = X_train_scaled_smote[knn_features] * feature_weights, y_train_smote
else:
    X_fit, y_fit = X_train_scaled[knn_features] * feature_weights, y_train
X_test_knn = X_test_scaled[knn_features] * feature_weights

t0 = time.time()
knn_model = KNeighborsClassifier(**knn_best_params, n_jobs=N_JOBS).fit(X_fit, y_fit)
knn_test_proba = knn_model.predict_proba(X_test_knn)[:, 1]
print(f"Final KNN fitted on {len(X_fit):,} rows x {len(knn_features)} features and scored on test in {time.time() - t0:.1f}s")

# Out-of-fold probabilities on the full training set (used only for threshold selection)
t0 = time.time()
knn_oof = oof_knn(knn_best_params, X_train_scaled[knn_features], y_train, train_groups_arr,
                  use_smote=knn_use_smote, power=knn_power)
print(f"KNN out-of-fold predictions on full train in {time.time() - t0:.1f}s")
print("KNN OOF metrics @0.5:", {k: round(v, 4) for k, v in evaluate(y_train, knn_oof).items()})

### COST-BASED DECISION THRESHOLD

The dataset has no cost figures, so the following **assumption** is stated:

- `COST_FN = 2`: a missed cancellation loses the room revenue for the stay.
- `COST_FP = 1`: a false alarm costs outreach effort, guest friction and a small over-booking risk.

Change these two constants to test other cost ratios. The threshold is chosen on **out-of-fold training predictions** and then applied unchanged to the test set.

In [ ]:
COST_FN = 2.0
COST_FP = 1.0
thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

rows = []
for t in thresholds:
    pred = (knn_oof >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_train, pred).ravel()
    rows.append({"threshold": t, "cost": COST_FN * fn + COST_FP * fp, "f1": f1_score(y_train, pred)})
thr_table = pd.DataFrame(rows)
knn_threshold = float(thr_table.loc[thr_table["cost"].idxmin(), "threshold"])
f1_thr = float(thr_table.loc[thr_table["f1"].idxmax(), "threshold"])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thr_table["threshold"], thr_table["cost"] / len(y_train), color="#2b5c8f", label="Avg cost per booking")
ax.axvline(knn_threshold, color="crimson", ls="--", label=f"Min-cost threshold = {knn_threshold:.2f}")
ax.axvline(0.5, color="grey", ls=":", label="Default 0.50")
ax2 = ax.twinx()
ax2.plot(thr_table["threshold"], thr_table["f1"], color="#2ca02c", alpha=0.6, label=f"F1 (max at {f1_thr:.2f})")
ax2.set_ylabel("F1")
ax.set_title("KNN: Cost vs Threshold (OOF train)", fontweight="bold")
ax.set_xlabel("Decision threshold")
ax.set_ylabel(f"Cost (FN={COST_FN:g}, FP={COST_FP:g}) / booking")
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=8, loc="upper center")
plt.tight_layout()
plt.show()
print(f"Chosen KNN threshold: {knn_threshold:.2f}")

### TEST-SET EVALUATION

In [ ]:
test_rows = []
for label, t in [("0.50", 0.5), (f"cost-opt {knn_threshold:.2f}", knn_threshold)]:
    tn, fp, fn, tp = confusion_matrix(y_test, (knn_test_proba >= t).astype(int)).ravel()
    test_rows.append({"Threshold": label, **evaluate(y_test, knn_test_proba, t),
                      "Avg cost / booking": (COST_FN * fn + COST_FP * fp) / len(y_test)})
knn_test_results = pd.DataFrame(test_rows).set_index("Threshold")

print(classification_report(y_test, (knn_test_proba >= knn_threshold).astype(int),
                            target_names=["Not canceled", "Canceled"], digits=4))
knn_test_results.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, (knn_test_proba >= knn_threshold).astype(int), ax=axes[0], cmap="Blues", colorbar=False,
    display_labels=["Not canceled", "Canceled"])
axes[0].set_title(f"KNN – Test Confusion Matrix (t={knn_threshold:.2f})", fontweight="bold")

fpr, tpr, _ = roc_curve(y_test, knn_test_proba)
axes[1].plot(fpr, tpr, color="#ff7f0e", label=f"KNN (AUC = {roc_auc_score(y_test, knn_test_proba):.3f})")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set(xlabel="False Positive Rate", ylabel="True Positive Rate")
axes[1].set_title("ROC Curve – Test Set", fontweight="bold")
axes[1].legend(loc="lower right")

prec, rec, _ = precision_recall_curve(y_test, knn_test_proba)
axes[2].plot(rec, prec, color="#ff7f0e", label=f"KNN (AP = {average_precision_score(y_test, knn_test_proba):.3f})")
axes[2].axhline(y_test.mean(), color="k", ls="--", lw=1, label=f"No-skill ({y_test.mean():.3f})")
axes[2].set(xlabel="Recall", ylabel="Precision")
axes[2].set_title("Precision–Recall Curve – Test Set", fontweight="bold")
axes[2].legend(loc="lower left")
plt.tight_layout()
plt.show()

### TIME-BASED CHECK (train on earlier arrivals, test on later ones)

The tuned KNN is refit on training rows arriving **before 2017-01-01** and evaluated on test rows arriving **on or after 2017-01-01**. Feature weights are recomputed from the earlier rows only. The scaler from Part B was fitted on the whole training set, so this is an approximate check.

In [ ]:
cutoff = pd.Timestamp("2017-01-01")
early_tr = (meta_train["arrival_date"] < cutoff).to_numpy()
late_te = (meta_test["arrival_date"] >= cutoff).to_numpy()
print(f"Temporal train rows: {early_tr.sum():,} | temporal test rows: {late_te.sum():,}")

X_k, y_k = X_train_scaled.loc[early_tr, knn_features], y_train[early_tr]
w_time = mi_to_weights(mi_scores(X_k, y_k), knn_power) if knn_power > 0 else np.ones(len(knn_features))
if knn_use_smote:
    X_k, y_k = smote_nc(X_k, y_k, X_k.columns.isin(discrete_cols), k=5, random_state=RANDOM_STATE)
knn_time = KNeighborsClassifier(**knn_best_params, n_jobs=N_JOBS).fit(X_k * w_time, y_k)
knn_time_proba = knn_time.predict_proba(X_test_scaled.loc[late_te, knn_features] * w_time)[:, 1]

time_results = pd.DataFrame([
    {"Split": "Random (group-aware)", **evaluate(y_test, knn_test_proba, knn_threshold)},
    {"Split": "Temporal (<2017 → 2017)", **evaluate(y_test[late_te], knn_time_proba, knn_threshold)},
]).set_index("Split")
time_results.round(4)

### FAIRNESS CHECK – error rates by country group

`country` is a proxy for nationality. Recall (cancellations caught) and false-positive rate (guests wrongly flagged) are compared across the grouped countries on the test set.

In [ ]:
def group_error_rates(proba, threshold, groups):
    pred = (proba >= threshold).astype(int)
    rows = []
    for g in groups.unique():
        m = (groups == g).to_numpy()
        yt, yp = y_test.to_numpy()[m], pred[m]
        tp = ((yt == 1) & (yp == 1)).sum(); fn = ((yt == 1) & (yp == 0)).sum()
        fp = ((yt == 0) & (yp == 1)).sum(); tn = ((yt == 0) & (yp == 0)).sum()
        rows.append({"country_group": g, "n": m.sum(), "actual_cancel_rate": yt.mean(),
                     "predicted_cancel_rate": yp.mean(),
                     "recall": tp / (tp + fn) if (tp + fn) else np.nan,
                     "FPR": fp / (fp + tn) if (fp + tn) else np.nan})
    return pd.DataFrame(rows).sort_values("n", ascending=False).set_index("country_group")

fair_knn = group_error_rates(knn_test_proba, knn_threshold, meta_test["country_grouped"])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fair_knn["recall"].plot(kind="bar", ax=axes[0], color="#ff7f0e")
axes[0].set_title("KNN Recall by Country Group (Test)", fontweight="bold")
fair_knn["FPR"].plot(kind="bar", ax=axes[1], color="#c0504d")
axes[1].set_title("KNN False-Positive Rate by Country Group (Test)", fontweight="bold")
for ax in axes:
    ax.set_xlabel("")
plt.tight_layout()
plt.show()
fair_knn.round(3)

### SAVE MODEL & RESULTS

To score a new booking: apply the Part B preprocessing, keep the columns in `features`, multiply by `feature_weights`, then call `model.predict_proba`.

In [ ]:
os.makedirs("models", exist_ok=True)
preprocessors = joblib.load("models/preprocessors.joblib")
joblib.dump({"model": knn_model, "params": knn_best_params, "uses_smote": knn_use_smote,
             "threshold": knn_threshold, "features": knn_features,
             "feature_weights": feature_weights, "feature_weight_power": knn_power,
             "preprocessors": preprocessors},
            "models/knn_model.joblib", compress=3)   # compressed: KNN stores the whole training set
knn_test_results.to_csv("models/knn_test_results.csv")
optimisation_steps.to_csv("models/knn_optimisation_steps.csv")

print("Best KNN params:", knn_best_params, "| feature-weighting power:", knn_power,
      "| SMOTE-NC:", knn_use_smote, "| threshold:", knn_threshold)
print("Saved: models/knn_model.joblib, models/knn_test_results.csv, models/knn_optimisation_steps.csv")